# 1. Palabra real

Se construye la palabra literal de cinco factores. Los roles tipados registran que los bloques del artículo ya contienen las interfaces auxiliares.

In [1]:
from pathlib import Path
import sys
repo_root = Path.cwd()
if not (repo_root / 'src').is_dir():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / 'src'))
import sympy as sp  # noqa: E402
import symbolic_operator_calculus as soc  # noqa: E402
gamma = sp.Symbol('gamma', positive=True)
lam = sp.Symbol('lambda', real=True)
x = sp.Symbol('x', positive=True)
context = soc.AssumptionContext()
space = soc.WeightedLpSpace('X_delta', sp.Integer(2), sp.Integer(0), context, 'notebook weighted space')
dilation = soc.WeightedDilationOperator(soc.OperatorAtom('V_gamma'), gamma, space, space, context, 'normalized dilation', evidence=('norm calculation',))
inverse = dilation.inverse(soc.OperatorAtom('V_gamma_inverse'))
freq = soc.mellin_frequency(lam)
radial = soc.output_spatial_variable(x)
domain = soc.MellinSymbolDomain(frequency=freq, complex_domain=soc.ComplexDomain.real_line(lam), spatial_variables=(radial,), assumption_context=context, singular_set=soc.SingularSet(), description='notebook domain', evidence=('typed domain',))
expression = soc.MellinExpression(x + lam, domain, (freq, radial), evidence=('typed symbol',), description='r11')
symbol = soc.MellinSymbol(expression, soc.MellinSymbolDependency.SPACE_FREQUENCY, 'r11')
pdo = soc.MellinPseudodifferentialOperator(soc.OperatorAtom('Op_r11'), symbol, space, space, 'tilde-E', 'regularizer PDO', True, ('radial scaling',), evidence=('bounded',))
regularizer = soc.RegularizerOperator(soc.ExactBlock('A', 1, 1), soc.OperatorAtom('R1', kind='formal_regularizer'))
representation = soc.RegularizerMellinRepresentation(regularizer, pdo, soc.RegularizerRepresentationStatus.CERTIFIED_EXACT, (), 'article R1 interface', ('article lines 161-168',), space)
mellin_core = soc.TransportedMellinCore(representation, soc.OperatorAtom('Phi_delta_inverse'), soc.OperatorAtom('Phi_delta'), 'transported core', ('article identity',))
left_wh = soc.WienerHopfOperator(soc.OperatorAtom('W_left'), lam + 2, lam, space, space, 'Fourier multipliers', 'left WH', True, ('scale stable',), evidence=('bounded',))
right_wh = soc.WienerHopfOperator(soc.OperatorAtom('W_right'), lam - 2, lam, space, space, 'Fourier multipliers', 'right WH', True, ('scale stable',), evidence=('bounded',))
synthetic_left_rep = soc.SchurBlockRepresentation(soc.ExactBlock('A', 2, 1), soc.OperatorAtom('B21_primitive'), (left_wh,), soc.DerivationRelationKind.EXACT_EQUALITY, 'synthetic exact left', (), ('evidence',))
synthetic_right_rep = soc.SchurBlockRepresentation(soc.ExactBlock('A', 1, 2), soc.OperatorAtom('B12_primitive'), (right_wh,), soc.DerivationRelationKind.EXACT_EQUALITY, 'synthetic exact right', (), ('evidence',))
synthetic = soc.factor_full_first_schur_word(left_block=soc.FullWordBlock(synthetic_left_rep), left_auxiliary=dilation, mellin_core=mellin_core, right_auxiliary=inverse, right_block=soc.FullWordBlock(synthetic_right_rep), assumptions=context)
cutoff = soc.SpatialCutoffOperator(soc.OperatorAtom('M_chi_infinity'), sp.Function('chi_infinity')(x), x, space, space, 'x >= 1', 'infinity; x >= 2', 'zero; x <= 1', 'half-line x', sp.Integer(1), 'article cutoff', ('verified localization',))
cutoff_trace = soc.conjugate_cutoff_by_dilation(dilation, cutoff, inverse)
T1 = soc.OperatorAtom('T1_minus')
Z1inv = soc.OperatorAtom('Z1_inverse')
left_article_rep = soc.SchurBlockRepresentation(soc.ExactBlock('A', 2, 1), soc.OperatorAtom('B21_minus'), (soc.OperatorAtom('row_difference_21'), soc.OperatorAtom('W21_localized'), T1), soc.DerivationRelationKind.EXACT_EQUALITY, 'article B21 minus', (), ('article lines 382-390',))
right_article_rep = soc.SchurBlockRepresentation(soc.ExactBlock('A', 1, 2), soc.OperatorAtom('B12_plus'), (Z1inv, soc.OperatorAtom('row_difference_12'), soc.OperatorAtom('W12_localized')), soc.DerivationRelationKind.EXACT_EQUALITY, 'article B12 plus', (), ('article lines 391-399',))
article_left = soc.FullWordBlock(left_article_rep, (soc.RegularizerInterfaceRole.LEFT_AUXILIARY,))
article_right = soc.FullWordBlock(right_article_rep, (soc.RegularizerInterfaceRole.RIGHT_AUXILIARY,))
article_result = soc.factor_full_first_schur_word(left_block=article_left, left_auxiliary=T1, mellin_core=mellin_core, right_auxiliary=Z1inv, right_block=article_right, assumptions=context)
print('real word:', ' '.join(atom.name for atom in article_result.original_word.factors))

real word: B21_minus T1_minus R1 Z1_inverse B12_plus


# 2. AST factor por factor

La expansión conserva las duplicaciones; no simplifica $\mathcal T_{1,-}^2$ ni $Z_1^{-2}$.

In [2]:
print('expanded:', ' '.join(atom.name for atom in article_result.expanded_original_word.factors))

expanded: row_difference_21 W21_localized T1_minus T1_minus R1 Z1_inverse Z1_inverse row_difference_12 W12_localized


# 3. Núcleo izquierdo

El bloque artículo-etiquetado ya contiene $\mathcal T_{1,-}$; agregar el argumento auxiliar repite la interfaz.

In [3]:
print('left core:', article_result.left_core.status.value)

left core: BLOCKED


# 4. Núcleo Mellin

El transporte $\Phi_\delta^{-1}\operatorname{Op}_M(r_{1,1})\Phi_\delta$ permanece explícito.

In [4]:
print('Mellin core:', ' '.join(atom.name for atom in mellin_core.transported_word.factors))

Mellin core: Phi_delta_inverse Op_r11 Phi_delta


# 5. Núcleo derecho

El bloque del artículo ya inicia con $Z_1^{-1}$; el argumento derecho introduce la segunda copia.

In [5]:
print('right core:', article_result.right_core.status.value)

right core: BLOCKED


# 6. Cortes reescalados

La covariancia exacta crea un objeto nuevo con símbolo $\chi_\infty(\gamma x)$ y escala radial $\gamma$.

In [6]:
print('scaled cutoff:', sp.sstr(cutoff_trace.scaled_cutoff.cutoff_symbol))
print('radial scale:', cutoff_trace.scaled_cutoff.radial_scale)

scaled cutoff: chi_infinity(gamma*x)
radial scale: gamma


# 7. Intento de factorización

El caso sintético prueba la ruta positiva; la palabra literal prueba el cierre seguro.

In [7]:
print('synthetic:', synthetic.status.value)
print('article-labelled:', article_result.status.value)
print('prior cancellation delegated:', synthetic.cancellation_result is not None)

synthetic: FULL_WORD_EXACT_FACTORIZATION
article-labelled: BLOCKED_BY_REGULARIZER_INTERFACE
prior cancellation delegated: True


# 8. Estado final

La clasificación única para la instancia solicitada se imprime sin promover el ejemplo sintético.

In [8]:
print('FINAL:', article_result.status.value)

FINAL: BLOCKED_BY_REGULARIZER_INTERFACE


# 9. Factores bloqueantes

Los roles se detectan por tipos y enums, no por coincidencias de cadenas.

In [9]:
print('blocking factor types:', ' '.join(type(item).__name__ for item in article_result.blocking_factors))

blocking factor types: FullWordBlock OperatorAtom OperatorAtom FullWordBlock


# 10. Obligaciones

Las siete obligaciones mínimas sobreviven en el resultado bloqueado.

In [10]:
for obligation in article_result.proof_obligations:
    print(obligation.key)

left_core_factorization
right_core_factorization
cutoff_replacement_mod_compacts
regularizer_mellin_representation
wh_mellin_wh_sandwich_membership
fredholm_algebra_membership
schur_correction_symbol


# 11. Salida LaTeX

La salida incluye palabra, estado y obligaciones; no contiene una igualdad candidata en el caso bloqueado.

In [11]:
print('latex status present:', r'BLOCKED\_BY\_REGULARIZER\_INTERFACE' in article_result.latex)
print('latex obligations present:', r'left\_core\_factorization' in article_result.latex)
print('factorized word:', article_result.factorized_word)

latex status present: True
latex obligations present: True
factorized word: None
